# Проект по SQL

Коронавирус застал мир врасплох, изменив привычный порядок вещей. На какое-то время жители городов перестали выходить на улицу, посещать кафе и торговые центры. Зато стало больше времени для книг. Это заметили стартаперы — и бросились создавать приложения для тех, кто любит читать.

Ваша компания решила быть на волне и купила крупный сервис для чтения книг по подписке. **Ваша первая задача** как аналитика — **проанализировать базу данных**.

В ней — информация о книгах, издательствах, авторах, а также пользовательские обзоры книг. Эти данные помогут сформулировать ценностное предложение для нового продукта.

## Исследуйте таблицы — выведите первые строки

In [1]:
# импортируем библиотеки
import pandas as pd
from sqlalchemy import text, create_engine

In [2]:
# устанавливаем параметры
db_config = {'user': 'praktikum_student', # имя пользователя
'pwd': 'Sdf4$2;d-d30pp', # пароль
'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
'port': 6432, # порт подключения
'db': 'data-analyst-final-project-db'} # название базы данных
connection_string = 'postgresql://{user}:{pwd}@{host}:{port}/{db}'.format(**db_config)
# сохраняем коннектор
engine = create_engine(connection_string, connect_args={'sslmode':'require'})
# чтобы выполнить SQL-запрос, используем Pandas
query = '''SELECT * FROM books LIMIT 5'''
con=engine.connect()
pd.io.sql.read_sql(sql=text(query), con = con)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


In [3]:
query = '''SELECT * FROM authors LIMIT 5'''
con=engine.connect()
pd.io.sql.read_sql(sql=text(query), con = con)

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


In [4]:
query = '''SELECT * FROM publishers LIMIT 5'''
con=engine.connect()
pd.io.sql.read_sql(sql=text(query), con = con)

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


In [5]:
query = '''SELECT * FROM ratings LIMIT 5'''
con=engine.connect()
pd.io.sql.read_sql(sql=text(query), con = con)

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


In [6]:
query = '''SELECT * FROM reviews LIMIT 5'''
con=engine.connect()
pd.io.sql.read_sql(sql=text(query), con = con)

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


## Сделайте по одному SQL-запросу для решения каждого задания

### Посчитайте, сколько книг вышло после 1 января 2000 года

In [7]:
query1 = '''SELECT COUNT(*) AS books_after_2000
FROM books
WHERE publication_date > '2000-01-01';'''
con1=engine.connect()
pd.io.sql.read_sql(sql=text(query1), con = con1)

,books_after_2000
0,819


### Для каждой книги посчитайте количество обзоров и среднюю оценку

In [8]:
query2 = '''SELECT
    b.book_id,
    b.title,
    COUNT(DISTINCT r.review_id) AS rvw_count, 
    AVG(rt.rating) AS avg_rating              
FROM
    books b
LEFT JOIN reviews r ON b.book_id = r.book_id     
LEFT JOIN ratings rt ON b.book_id = rt.book_id  
GROUP BY
    b.book_id, b.title
ORDER BY
    rvw_count DESC;'''
con2=engine.connect()
pd.io.sql.read_sql(sql=text(query2), con = con2)

,book_id,title,rvw_count,avg_rating
0,948,Twilight (Twilight #1),7,3.662500
1,963,Water for Elephants,6,3.977273
2,734,The Glass Castle,6,4.206897
3,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.414634
4,695,The Curious Incident of the Dog in the Night-Time,6,4.081081
...,...,...,...,...
995,83,Anne Rice's The Vampire Lestat: A Graphic Novel,0,3.666667
996,808,The Natural Way to Draw,0,3.000000
997,672,The Cat in the Hat and Other Dr. Seuss Favorites,0,5.000000
998,221,Essential Tales and Poems,0,4.000000


### Определите издательство, которое выпустило наибольшее число книг толще 50 страниц — так вы исключите из анализа брошюры

In [9]:
query3 = '''SELECT
    p.publisher_id,
    p.publisher,
    COUNT(b.book_id) AS book_count
FROM
    books b
JOIN publishers p ON b.publisher_id = p.publisher_id
WHERE
    b.num_pages > 50   
GROUP BY
    p.publisher_id, p.publisher     
ORDER BY
    book_count DESC   
LIMIT 1;'''
con3=engine.connect()
pd.io.sql.read_sql(sql=text(query3), con = con3)

,publisher_id,publisher,book_count
0,212,Penguin Books,42


### Определите автора с самой высокой средней оценкой книг — учитывайте только книги с 50 и более оценками

In [10]:
query4 = '''WITH book_ratings AS (
    SELECT
        b.book_id,
        b.author_id,
        AVG(rt.rating) AS avg_rating,
        COUNT(rt.rating_id) AS rating_count
    FROM
        books b
    JOIN ratings rt ON b.book_id = rt.book_id
    GROUP BY
        b.book_id, b.author_id
    HAVING
        COUNT(rt.rating_id) >= 50
),
author_avg_ratings AS (
    SELECT
        a.author_id,
        a.author,
        AVG(br.avg_rating) AS author_avg_rating
    FROM
        book_ratings br
    JOIN authors a ON br.author_id = a.author_id
    GROUP BY
        a.author_id, a.author
)
SELECT
    author_id,
    author,
    author_avg_rating
FROM
    author_avg_ratings
ORDER BY
    author_avg_rating DESC
LIMIT 1;'''
con4=engine.connect()
pd.io.sql.read_sql(sql=text(query4), con = con4)

,author_id,author,author_avg_rating
0,236,J.K. Rowling/Mary GrandPré,4.283844


### Посчитайте среднее количество обзоров от пользователей, которые поставили больше 48 оценок

In [11]:
query5 = '''WITH user_ratings AS (
    SELECT
        username,
        COUNT(rating_id) AS rtg_count
    FROM
        ratings
    GROUP BY
        username
    HAVING
        COUNT(rating_id) > 48
),
user_reviews AS (
    SELECT
        r.username,
        COUNT(r.review_id) AS rvw_count
    FROM
        reviews r
    JOIN user_ratings ur ON r.username = ur.username
    GROUP BY
        r.username
)
SELECT
    AVG(rvw_count) AS avg_rvws_per_user
FROM
    user_reviews;'''
con5=engine.connect()
pd.io.sql.read_sql(sql=text(query5), con = con5)

,avg_rvws_per_user
0,24.0


## Опишите выводы по каждой из решённых задач

### Посчитайте, сколько книг вышло после 1 января 2000 года

Видим **819 книг**, вышедших после указанной даты.

### Для каждой книги посчитайте количество обзоров и среднюю оценку

Добавили нужные столбцы и расчитали данные в них в каждой из строк.

### Определите издательство, которое выпустило наибольшее число книг толще 50 страниц — так вы исключите из анализа брошюры

Больше всего книг выпустило издание **Penguin Books** с 42 книгами.

### Определите автора с самой высокой средней оценкой книг — учитывайте только книги с 50 и более оценками

Автор с самым высоким рейтингом, получившим 50 отзывов или более, стал **J.K. Rowling/Mary GrandPré** со средней оценкой 4.28

### Посчитайте среднее количество обзоров от пользователей, которые поставили больше 48 оценок

Если брать пользователей с 48 оценками и более, то в среднем количество обзоров от них составило **24 обзора** на пользователя.